## Phase 3 — Validate Labeler and Calibrate (CPU)

### Cell 1 — Setup + Load

In [ ]:
import os, sys; sys.path.insert(0, ".")
os.environ["OPENAI_API_KEY"] = "<from secret>"
from sac.kqa_loader import load_physician_nli
from sac.cache import load_claims
from sac.validate import labeler_agreement, score_auroc, gate2_pass, unverifiable_count
from sac.openai_judge import OpenAIJudge
from sac.crc import crc_calibrate, apply_global, retention, realized_risk_marginal
import numpy as np

claims = load_claims("claims_phase2.jsonl")

### Cell 2 — Gate 1 (labeler vs physician NLI, automated)

In [ ]:
pairs = load_physician_nli("kqa_physician_nli.jsonl")   # confirm path/fields first
g1 = labeler_agreement(pairs, OpenAIJudge(model="gpt-4o"))
print(f"GATE 1  labeler accuracy={g1['accuracy']:.3f}  kappa={g1['kappa']:.3f}  n={g1['n']}")

### Cell 3 — Gate 2 (score AUROC, the Stage 1 success criterion)

In [ ]:
au = score_auroc(claims)
print(f"GATE 2  P(true) AUROC={au:.3f}  ->  {'PASS' if gate2_pass(au) else 'FAIL (<0.7, fix score)'}")
print("unverifiable claims (excluded from calibration):", unverifiable_count(claims))

### Cell 4 — Global CRC sanity on real claims (reuses Stage 0 method)

In [ ]:
cal = [c for c in claims if c.label in (0, 1)]           # drop unverifiable
np.random.seed(0)
idx = np.random.permutation(len(cal)); k = len(cal)//2
calib = [cal[i] for i in idx[:k]]; test = [cal[i] for i in idx[k:]]
lam = crc_calibrate([c.confidence for c in calib], [c.label for c in calib], alpha=0.10)
kept = apply_global(test, lam)
print(f"global lambda={lam:.3f}  test marginal risk={realized_risk_marginal(test, kept):.3f}  "
      f"retention={retention(test, kept):.3f}")